# 02 · Preprocessing — Vehicle Type (Classification)
No missing values or categories. Two steps: drop duplicates, standardise for logistic regression.

## Setup
Make `src/` importable and load the shared config (seed, paths).

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

# Make `src` importable: repo root locally, or an uploaded Kaggle dataset containing src/
for p in [Path.cwd(), *Path.cwd().parents, *(f.parent.parent for f in Path("/kaggle/input").glob("**/src/config.py"))]:
    if (p / "src" / "config.py").exists():
        sys.path.insert(0, str(p.resolve()))
        break

from src import config

Load raw and de-duplicated data.

In [2]:
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from src import evaluate
from src.classification import preprocess as pp

TASK = "classification"
pd.set_option("display.width", 160)

raw, _, _ = config.load(TASK)
FEATURES = [c for c in raw.columns if c not in (pp.ID, pp.TARGET)]
X_raw, y_raw = raw[FEATURES], raw[pp.TARGET]
X, y, X_test, test_ids, sample_sub = pp.load_clean()
print("raw train:", X_raw.shape, "| after dropping duplicates:", X.shape)

raw train: (10000, 20) | after dropping duplicates: (7500, 20)


## Duplicates leak into CV

In [3]:
key = X_raw.astype(str).agg("|".join, axis=1).to_numpy()
leaked = []
for tr, va in evaluate._splitter(TASK).split(X_raw, y_raw):
    leaked.append(np.isin(key[va], key[tr]).mean())
print("share of validation rows with an identical row in the training part, per fold:", np.round(leaked, 3))

share of validation rows with an identical row in the training part, per fold: [0.392 0.386 0.394 0.392 0.404]


- ~40% of validation rows have an identical row in the training part.

CV accuracy with and without duplicates.

In [4]:
models = {
    "full tree": DecisionTreeClassifier(random_state=config.SEED),
    "random forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=config.SEED),
    "logistic (scaled)": make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)),
    "LDA": LinearDiscriminantAnalysis(),
}
rows = []
for name, m in models.items():
    with_dups = evaluate.cross_validate(m, X_raw, y_raw, TASK, "", log=False)
    clean = evaluate.cross_validate(m, X, y, TASK, "", log=False)
    rows.append({"model": name, "CV with duplicates": with_dups.accuracy.mean(), "CV de-duplicated": clean.accuracy.mean(),
                 "inflation (points)": 100 * (with_dups.accuracy.mean() - clean.accuracy.mean()),
                 "train accuracy": clean.train_accuracy.mean()})
dup_table = pd.DataFrame(rows).set_index("model")
display(dup_table.round(4))

,CV with duplicates,CV de-duplicated,inflation (points),train accuracy
model,,,,
full tree,0.8957,0.8241,7.1567,1.0000
random forest,0.9292,0.8825,4.6667,1.0000
logistic (scaled),0.8957,0.8941,0.1567,0.8960
LDA,0.8930,0.8924,0.0600,0.8943


- Full tree +7.2 points, Random Forest +4.7, logistic +0.2, so duplicates are removed before CV.

## Standardisation: which models need it?

In [5]:
def compare(name, model):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        t = time.time()
        raw_acc = evaluate.cross_validate(model, X, y, TASK, "", log=False).accuracy.mean()
        raw_time = time.time() - t
    not_converged = sum("converge" in str(x.message).lower() for x in w)
    scaled_acc = evaluate.cross_validate(make_pipeline(StandardScaler(), model), X, y, TASK, "", log=False).accuracy.mean()
    return {"model": name, "unscaled": raw_acc, "scaled": scaled_acc, "difference (points)": 100 * (scaled_acc - raw_acc),
            "unscaled fit time (s)": round(raw_time, 1), "folds not converged (unscaled)": not_converged}

scale_table = pd.DataFrame([
    compare("decision tree", DecisionTreeClassifier(random_state=config.SEED)),
    compare("Gaussian NB", GaussianNB()),
    compare("LDA", LinearDiscriminantAnalysis()),
    compare("logistic C=0.1 (max_iter=1000)", LogisticRegression(C=0.1, max_iter=1000)),
    compare("logistic C=0.01 (max_iter=1000)", LogisticRegression(C=0.01, max_iter=1000)),
]).set_index("model")
display(scale_table.round(4))

,unscaled,scaled,difference (points),unscaled fit time (s),folds not converged (unscaled)
model,,,,,
decision tree,0.8241,0.8247,0.0533,0.4,0
Gaussian NB,0.8465,0.8473,0.0800,0.1,0
LDA,0.8924,0.8924,0.0000,0.1,0
logistic C=0.1 (max_iter=1000),0.8755,0.8941,1.8667,5.2,5
logistic C=0.01 (max_iter=1000),0.8761,0.8909,1.4800,5.2,5


- Only logistic regression is affected: −1.9 points and no convergence without scaling.

Feature effect (|weight| × std) with and without scaling.

In [6]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    unscaled = LogisticRegression(C=0.1, max_iter=1000).fit(X, y)
scaled = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(X, y)
effect = pd.DataFrame({
    "feature std": X.std(),
    "effect unscaled (|w|·std)": np.abs(unscaled.coef_).max(axis=0) * X.std().to_numpy(),
    "effect scaled (|w|)": np.abs(scaled[-1].coef_).max(axis=0),
}).sort_values("feature std")
display(effect.round(3))

,feature std,effect unscaled (|w|·std),effect scaled (|w|)
drag_coefficient,0.043,0.000,0.141
turning_radius_m,0.523,0.004,0.189
engine_displacement_l,0.611,0.001,0.560
engine_cylinders,1.502,0.006,0.105
acceleration_0_100_kmh_s,2.258,0.045,1.476
highway_fuel_efficiency_kmpl,2.499,0.159,0.365
city_fuel_efficiency_kmpl,2.555,0.137,0.272
braking_distance_m,7.231,0.472,0.477
fuel_tank_capacity_l,14.475,1.228,0.325
battery_capacity_kwh,16.936,1.981,0.661


## Clipped values are kept

In [7]:
at_floor = X.ground_clearance_mm == X.ground_clearance_mm.min()
display(pd.DataFrame({
    "at 110 mm floor": y[at_floor].map(pp.CLASS_NAMES).value_counts(normalize=True),
    "all rows": y.map(pp.CLASS_NAMES).value_counts(normalize=True),
}).round(3))
print(f"rows at floor: {at_floor.sum():,} of {len(X):,}")

,at 110 mm floor,all rows
vehicle_class,,
SUV,0.081,0.332
Sedan,0.171,0.339
Sports Car,0.748,0.329


rows at floor: 1,731 of 7,500


- 75% of cars at the 110 mm floor are Sports Cars, so the floor value carries signal.